In [14]:
from pathlib import Path
import os
import json
import numpy as np

hartree_to_ev = 27.2114


In [ ]:
class PropertiesRedox:
    """Properties from JSON file from Orca Vertical (no opt) redox calculations"""

    def __init__(self, path):
        """this class reads an orca json file from a given path and extracts properties
        Args:
            path (Path): Path to the orca property.json file
        The class is initialized by reading the json file
        """
        
        self.path = path

        self.results = {}

        if 'singlepoint.property.json' in os.listdir(path):
            self.neutraljson = json.load(open(path + 'singlepoint.property.json'))
        else:
            print("No singlepoint.property.json file found in the specified path. Results dictionary will be empty.")
            self.neutraljson = {}

        if 'oxidation.property.json' in os.listdir(path):
            self.oxidationjson = json.load(open(path + 'oxidation.property.json'))
        else:
            print("No oxidation.property.json file found in the specified path. Results dictionary will be empty.")
            self.oxidationjson = {}

        if 'reduction.property.json' in os.listdir(path):
            self.reductionjson = json.load(open(path + 'reduction.property.json'))
        else:
            print("No reduction.property.json file found in the specified path. Results dictionary will be empty.")
            self.reductionjson = {}


        # initialize lists for atoms, bonds, angles, dihedrals
        self.atom_list = None

    def define_atom_list(self, atom_list):
        self.atom_list = atom_list

    def get_electron_affinity(self, units='eV'):
        """get vertical electron affinity from neutral and reduced species
        """
        if not self.neutraljson or not self.reductionjson:
            print("Neutral or reduction JSON data is missing. Cannot compute electron affinity.")
            return

        e_neutral = self.neutraljson['Geometries'][-1]['DFT_Energy']['finalEn']
        e_reduced = self.reductionjson['Geometries'][-1]['DFT_Energy']['finalEn']

        ea = e_neutral - e_reduced  # in Hartree

        # only units ev or hartree are allowed
        if units == 'eV':
            ea_converted = ea * hartree_to_ev
        elif units == 'Hartree':
            ea_converted = ea
        else:
            print("Invalid units specified. Use 'eV' or 'Hartree'.")
            return

        self.results['electron_affinity'] = ea_converted

    def get_ionization_potential(self, units='eV'):
        """get vertical ionization potential from neutral and oxidized species
        """
        if not self.neutraljson or not self.oxidationjson:
            print("Neutral or oxidation JSON data is missing. Cannot compute ionization potential.")
            return

        e_neutral = self.neutraljson['Geometries'][-1]['DFT_Energy']['finalEn']
        e_oxidized = self.oxidationjson['Geometries'][-1]['DFT_Energy']['finalEn']

        ip = e_oxidized - e_neutral  # in Hartree

        # only units ev or hartree are allowed
        if units == 'eV':
            ip_converted = ip * hartree_to_ev
        elif units == 'Hartree':
            ip_converted = ip
        else:
            print("Invalid units specified. Use 'eV' or 'Hartree'.")
            return

        self.results['ionization_potential'] = ip_converted

    def get_chemical_hardness(self, units='eV'):
        """get chemical hardness from ionization potential and electron affinity
        """
        if 'ionization_potential' not in self.results:
            self.get_ionization_potential(units=units)
        if 'electron_affinity' not in self.results:
            self.get_electron_affinity(units=units)

        if 'ionization_potential' in self.results and 'electron_affinity' in self.results:
            ip = self.results['ionization_potential']
            ea = self.results['electron_affinity']

            hardness = (ip - ea) / 2
            self.results['chemical_hardness'] = hardness
        else:
            print("Cannot compute chemical hardness due to missing IP or EA.")

    def get_electronegativity(self, units='eV'):
        """get electronegativity from ionization potential and electron affinity
        """
        if 'ionization_potential' not in self.results:
            self.get_ionization_potential(units=units)
        if 'electron_affinity' not in self.results:
            self.get_electron_affinity(units=units)

        if 'ionization_potential' in self.results and 'electron_affinity' in self.results:
            ip = self.results['ionization_potential']
            ea = self.results['electron_affinity']

            electronegativity = (ip + ea) / 2
            self.results['electronegativity'] = electronegativity
        else:
            print("Cannot compute electronegativity due to missing IP or EA.")

    def _get_mulliken_charge_set(self, at_idx):
        neutral_charge = self.neutraljson['Geometries'][-1]['Mulliken_Population_Analysis'][0]['AtomicCharges'][at_idx-1][0]
        oxidation_charge = self.oxidationjson['Geometries'][-1]['Mulliken_Population_Analysis'][0]['AtomicCharges'][at_idx-1][0]
        reduction_charge = self.reductionjson['Geometries'][-1]['Mulliken_Population_Analysis'][0]['AtomicCharges'][at_idx-1][0]
        return neutral_charge, oxidation_charge, reduction_charge
    
    def _get_loewdin_charge_set(self, at_idx):
        neutral_charge = self.neutraljson['Geometries'][-1]['Loewdin_Population_Analysis'][0]['AtomicCharges'][at_idx-1][0]
        oxidation_charge = self.oxidationjson['Geometries'][-1]['Loewdin_Population_Analysis'][0]['AtomicCharges'][at_idx-1][0]
        reduction_charge = self.reductionjson['Geometries'][-1]['Loewdin_Population_Analysis'][0]['AtomicCharges'][at_idx-1][0]
        return neutral_charge, oxidation_charge, reduction_charge
    
    def _get_hirshfeld_charge_set(self, at_idx):
        neutral_charge = self.neutraljson['Geometries'][-1]['Hirshfeld_Population_Analysis'][0]['AtomicCharges'][at_idx-1][0]
        oxidation_charge = self.oxidationjson['Geometries'][-1]['Hirshfeld_Population_Analysis'][0]['AtomicCharges'][at_idx-1][0]
        reduction_charge = self.reductionjson['Geometries'][-1]['Hirshfeld_Population_Analysis'][0]['AtomicCharges'][at_idx-1][0]
        return neutral_charge, oxidation_charge, reduction_charge
    
    def _get_gelpg_charge_set(self, at_idx):
        neutral_charge = self.neutraljson['Geometries'][-1]['GELPG_Population_Analysis'][0]['AtomicCharges'][at_idx-1][0]
        oxidation_charge = self.oxidationjson['Geometries'][-1]['GELPG_Population_Analysis'][0]['AtomicCharges'][at_idx-1][0]
        reduction_charge = self.reductionjson['Geometries'][-1]['GELPG_Population_Analysis'][0]['AtomicCharges'][at_idx-1][0]
        return neutral_charge, oxidation_charge, reduction_charge
    
    # def _get_npa_charge_set(self, at_idx):
    #     if not self.neutral_nbo_json or not self.oxidation_nbo_json or not self.reduction_nbo_json:
    #         print("NBO JSON data is missing. Cannot retrieve NPA charges.")
    #         return np.nan, np.nan, np.nan

    #     neutral_charge = self.neutral_nbo_json['natural_population_analysis']['atoms'][at_idx]["charge"]
    #     oxidation_charge = self.oxidation_nbo_json['natural_population_analysis']['atoms'][at_idx]["charge"]
    #     reduction_charge = self.reduction_nbo_json['natural_population_analysis']['atoms'][at_idx]["charge"]
    #     return neutral_charge, oxidation_charge, reduction_charge
    
    def get_atomic_fplus(self, method='mulliken'):
        """
        """
        if self.atom_list is None:
            print("Atom list is not defined. Please define it using define_atom_list().")
            return
        
        method_function = getattr(self, f"_get_{method}_charge_set")
        
        for at_idx in self.atom_list:
            neutral_charge, oxidation_charge, _ = method_function(at_idx)
            fplus = oxidation_charge - neutral_charge
            self.results[f"fplus_{method}_{at_idx}"] = fplus

    def get_atomic_fminus(self, method='mulliken'):
        """
        """
        if self.atom_list is None:
            print("Atom list is not defined. Please define it using define_atom_list().")
            return
        
        method_function = getattr(self, f"_get_{method}_charge_set")
        
        for at_idx in self.atom_list:
            neutral_charge, _, reduction_charge = method_function(at_idx)
            fminus = neutral_charge - reduction_charge
            self.results[f"fminus_{method}_{at_idx}"] = fminus

    def get_atomic_f0(self, method='mulliken'):
        """
        """
        if self.atom_list is None:
            print("Atom list is not defined. Please define it using define_atom_list().")
            return
        
        method_function = getattr(self, f"_get_{method}_charge_set")
        
        for at_idx in self.atom_list:
            _, oxidation_charge, reduction_charge = method_function(at_idx)
            f0 = (oxidation_charge - reduction_charge) / 2
            self.results[f"f0_{method}_{at_idx}"] = f0

    def get_all_f(self, method='mulliken'):
        self.get_atomic_f0(method=method)
        self.get_atomic_fplus(method=method)
        self.get_atomic_fminus(method=method)


In [31]:
P = PropertiesRedox(path='orca_exampleNi_2/')
P.get_atomic_f0()
P.define_atom_list([1, 5])
P.get_atomic_f0(method='hirshfeld')
P.get_all_f()
P.get_electronegativity()
P.results

No singlepoint.nbo_properties.json file found in the specified path. Cannot load NBO properties.
No oxidation.nbo_properties.json file found in the specified path. Cannot load NBO properties.
No reduction.nbo_properties.json file found in the specified path. Cannot load NBO properties.
Atom list is not defined. Please define it using define_atom_list().


{'f0_hirshfeld_1': 0.07743728624832147,
 'f0_hirshfeld_5': 0.03909438573277235,
 'f0_mulliken_1': 0.12828476644154058,
 'f0_mulliken_5': 0.04945127570123464,
 'fplus_mulliken_1': 0.1822177098490999,
 'fplus_mulliken_5': 0.05647467701868525,
 'fminus_mulliken_1': 0.07435182303398125,
 'fminus_mulliken_5': 0.04242787438378404,
 'ionization_potential': 5.799930854070606,
 'electron_affinity': 0.7178127299309666,
 'electronegativity': 3.2588717920007864}